# Maintenance: compact and clean up the Delta tables

Every pipeline run appends small files (the live job every few minutes during matches), and the dashboard
reads these tables straight from S3 with DuckDB, one HTTP request per file. `OPTIMIZE` rewrites each
table's current data into as few files as possible; `VACUUM` deletes files no longer referenced by any
version from the last 7 days (the default retention), so time travel within a week still works.
Runs at the end of the nightly `football-batch` job.

In [0]:
from delta.tables import DeltaTable
import pandas as pd

S3_BUCKET = "football-data-raw-01"
ROOTS = [f"s3://{S3_BUCKET}/silver", f"s3://{S3_BUCKET}/gold", f"s3://{S3_BUCKET}/live/live_delta"]

def delta_tables(root):
    for entry in dbutils.fs.ls(root):
        if entry.isDir() and any(child.name == "_delta_log/" for child in dbutils.fs.ls(entry.path)):
            yield entry.path.rstrip("/")

def file_count(path):
    return DeltaTable.forPath(spark, path).detail().select("numFiles").first()[0]

results = []
for root in ROOTS:
    for path in delta_tables(root):
        before = file_count(path)
        spark.sql(f"OPTIMIZE delta.`{path}`")
        spark.sql(f"VACUUM delta.`{path}`")
        results.append((path.replace(f"s3://{S3_BUCKET}/", ""), before, file_count(path)))

summary = pd.DataFrame(results, columns=["table", "files_before", "files_after"])
print(f"{len(summary)} tables: {summary.files_before.sum()} -> {summary.files_after.sum()} data files")
display(summary)